# Week 05 - Image Preprocessing I: Brightness and Geometric Transformations

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Apply **pixel brightness transformations**: contrast, brightness, gamma.
- Use **histogram equalization** and **CLAHE** to improve contrast.
- Perform **affine** transformations (translation, rotation, scaling, shear).
- Perform a **perspective** transformation and estimate a **homography** from 4 point correspondences.

### Two families of transformation
- **Point operations** change a pixel using only its own value: $g(x,y) = T[f(x,y)]$.
- **Geometric operations** move pixels to new locations: affine (3 DoF+ ) and perspective (8 DoF).

## 1. Setup

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, cmap=None):
    titles = titles or [""] * len(images)
    plt.figure(figsize=(5 * len(images), 5))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img, cmap=cmap or (None if img.ndim == 3 else "gray"))
        plt.title(titles[i]); plt.axis("off")
    plt.tight_layout(); plt.show()

img = cv2.imread("resources/images/test_image.jpeg")
print("Shape:", img.shape)

## 2. Guided example - brightness and contrast
$g = \alpha f + \beta$ where $\alpha$ controls **contrast** and $\beta$ controls **brightness**. This is `cv2.convertScaleAbs`.

In [ ]:
bright = cv2.convertScaleAbs(img, alpha=1.0, beta=60)   # +brightness
dark   = cv2.convertScaleAbs(img, alpha=1.0, beta=-60)  # -brightness
low_c  = cv2.convertScaleAbs(img, alpha=0.5, beta=0)    # -contrast
high_c = cv2.convertScaleAbs(img, alpha=1.8, beta=0)    # +contrast
show(img, bright, dark, low_c, high_c,
     titles=["Original", "beta=+60", "beta=-60", "alpha=0.5", "alpha=1.8"])

## 3. Guided example - gamma correction
Gamma is a **non-linear** point operation: $g = (f/255)^{1/\gamma} \times 255$.
- $\gamma < 1$ brightens shadows
- $\gamma > 1$ darkens highlights

This is essential when a camera's response is non-linear.

In [ ]:
def gamma_correct(image, gamma):
    inv = 1.0 / gamma
    table = (np.arange(256) / 255.0) ** inv * 255
    return cv2.LUT(image, table.astype(np.uint8))

g_low  = gamma_correct(img, 0.4)   # brighter
g_high = gamma_correct(img, 2.5)   # darker
show(g_low, img, g_high, titles=["gamma=0.4 (brighter)", "Original", "gamma=2.5 (darker)"])

## 4. Guided example - histogram equalization and CLAHE
Equalization spreads intensities to use the full range. **CLAHE** does it locally in tiles, avoiding the washed-out look of global equalization.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
eq = cv2.equalizeHist(gray)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(gray)

show(gray, eq, clahe, titles=["Original gray", "Global equalization", "CLAHE"])

## 5. Guided example - affine transformations
An affine transform keeps parallel lines parallel. It has **6 parameters** and is written as a $2\times3$ matrix used in `cv2.warpAffine`. A rotation uses `cv2.getRotationMatrix2D`.

In [ ]:
rows, cols = img.shape[:2]

# Translation matrix [1 0 tx; 0 1 ty]
M_translate = np.float32([[1, 0, 100], [0, 1, 50]])
translated = cv2.warpAffine(img, M_translate, (cols, rows))

# Rotation matrix about the centre
M_rotate = cv2.getRotationMatrix2D((cols / 2, rows / 2), 30, 1.0)
rotated = cv2.warpAffine(img, M_rotate, (cols, rows))

# Scale + shear build manually
M_shear = np.float32([[1, 0.4, 0], [0.1, 1, 0]])
sheared = cv2.warpAffine(img, M_shear, (int(cols * 1.4), int(rows * 1.2)))

show(translated, rotated, sheared, titles=["Translation", "Rotation 30 deg", "Shear + scale"])

## 6. Guided example - perspective and homography
A perspective transform maps a **quadrilateral to any quadrilateral** (8 parameters). We can estimate it from 4 point correspondences with `cv2.getPerspectiveTransform`. This is how you "scan" a document photographed at an angle.

In [ ]:
# --- Step 1: create a synthetic skewed version of the image ---
src = np.float32([[0, 0], [cols, 0], [cols, rows], [0, rows]])
dst = np.float32([[90, 60], [cols - 40, 20], [cols - 90, rows - 30], [40, rows - 80]])
H_warp = cv2.getPerspectiveTransform(src, dst)
skewed = cv2.warpPerspective(img, H_warp, (cols, rows))

# --- Step 2: recover the homography that undoes it ---
H_inv = cv2.getPerspectiveTransform(dst, src)
corrected = cv2.warpPerspective(skewed, H_inv, (cols, rows))

show(img, skewed, corrected, titles=["Original", "Perspective skew", "Corrected (undo)"])

> In practice you do not know the destination corners. In Week 8 you will detect them automatically with keypoints such as SIFT/ORB, then estimate the same homography for tasks like panorama stitching.

## 7. Exercise (complete the code)

1. Rotate the image by **-45 degrees** and scale it to **50%** in a single affine matrix.
   *Hint:* combine `cv2.getRotationMatrix2D((cx, cy), -45, 0.5)`.
2. Expand the output canvas so the rotated image is not cropped.
3. Display the result.

In [ ]:
# TODO: implement the combined rotation + scale


## 8. Challenge (independent)

Create a **perspective correction for a document**. Take a clean rectangle image of your own (or draw text on a white canvas), apply a strong perspective warp, then correct it using a homography estimated from the four corners. Measure the mean absolute difference between the original and the corrected image.

In [ ]:
# Your code here


## 9. Reflection
1. What is the difference between an affine and a perspective transformation in terms of degrees of freedom?
2. Why can a non-linear operation such as gamma not be written as a matrix multiply on the pixel values?
3. In a document scanner, why is a perspective transform required instead of an affine one?